# 09. Desarrollo e implementación del MVP web de InvisibleDogs Predict

Este notebook documenta la fase de **desarrollo, integración y despliegue del MVP web de InvisibleDogs Predict**, construido a partir de los modelos, representaciones visuales y artefactos validados en las fases anteriores del TFM.

El objetivo de esta etapa no es reentrenar modelos, sino **operacionalizar los resultados del trabajo experimental** y convertirlos en una aplicación web funcional orientada a dos perfiles de usuario:

- **Adoptante**, que puede explorar perros compatibles con sus preferencias, consultar fichas y utilizar mecanismos de búsqueda por características o similitud visual.
- **Protectora**, que puede consultar perfiles con información relativa al riesgo, la visibilidad, la completitud de las fichas y resultados derivados del análisis textual.

La aplicación actual se ha desarrollado con **Next.js, React y TypeScript**. El frontend y la lógica server-side asociada se encuentran en `frontend/`. Para la generación de embeddings a partir de nuevas fotografías se ha implementado, como servicio independiente, un componente Python basado en **FastAPI + DINOv2** localizado en `services/dinov2-inference/`.

El frontend se encuentra desplegado en **Microsoft Azure App Service** mediante un flujo automatizado de **GitHub Actions**.

**URL pública del MVP:**  
https://invisibledogs-predict-dfacg2ene0hrhmba.spaincentral-01.azurewebsites.net/


## 1. Objetivos del notebook

Los objetivos principales de esta fase son:

- definir la arquitectura funcional y tecnológica del MVP;
- reutilizar artefactos previamente generados y validados sin reentrenamiento durante el uso de la aplicación;
- implementar los recorridos diferenciados de **Adoptante** y **Protectora**;
- incorporar búsqueda y recuperación de candidatos mediante características estructuradas;
- incorporar búsqueda visual basada en representaciones DINOv2;
- integrar artefactos derivados de PetFinder para demostración histórica y análisis complementario;
- mantener una separación metodológica clara entre **Austin Animal Center**, **PetFinder** y **Tsinghua Dogs**;
- validar el funcionamiento del frontend mediante pruebas automatizadas;
- automatizar la compilación y el despliegue del frontend en **Azure App Service**;
- documentar de forma explícita qué componentes están operativos en el runtime actual y cuáles requieren todavía una integración adicional.

Esta fase representa la transición entre el trabajo experimental de Data Science y la operacionalización reproducible de los resultados obtenidos.


## 2. Principios metodológicos del MVP

El desarrollo mantiene los siguientes principios:

1. Los modelos no se reentrenan durante el uso de la aplicación.
2. Se reutilizan artefactos previamente generados y validados.
3. Las preferencias del adoptante se aplican sobre el catálogo preparado antes de ordenar o recuperar candidatos.
4. La similitud DINOv2 se interpreta como **parecido visual**, no como probabilidad de pertenencia a una raza.
5. El riesgo de larga estancia no determina por sí solo la compatibilidad entre un perro y un adoptante.
6. No se crean puntuaciones arbitrarias que mezclen similitud visual, riesgo y completitud.
7. **Austin Animal Center** constituye el núcleo predictivo principal del TFM para el riesgo de larga estancia.
8. **PetFinder** se utiliza como análisis complementario y como base histórica para parte del recorrido del adoptante y del análisis de adopción lenta.
9. **Tsinghua Dogs** se utiliza como referencia externa para validar representaciones visuales y recuperación por similitud.
10. La completitud de las fichas se trata como un indicador transparente basado en reglas, no como un nuevo modelo predictivo.


## 3. Arquitectura tecnológica implementada

La solución se ha estructurado en tres capas principales:

### 3.1. Aplicación web

El directorio `frontend/` contiene la aplicación desarrollada con:

- **Next.js 16**;
- **React 19**;
- **TypeScript**;
- **Tailwind CSS**;
- rutas server-side de Next.js para las operaciones que no deben exponerse directamente al cliente.

La aplicación incluye rutas diferenciadas para los recorridos de Adoptante y Protectora, además de pruebas automatizadas para Home, Adoptante y Protectora.

### 3.2. Servicio de inferencia visual

El directorio `services/dinov2-inference/` contiene un servicio Python basado en **FastAPI** cuya función es transformar una fotografía nueva en un embedding DINOv2:

```text
imagen
  -> preprocesado
  -> facebook/dinov2-small
  -> token CLS
  -> normalización L2
  -> vector float32 de 384 dimensiones
```

La búsqueda visual se completa en el servidor Next.js comparando el embedding de consulta con los embeddings PetFinder precomputados.

### 3.3. Artefactos preparados

El runtime web reutiliza artefactos derivados previamente, entre ellos:

- perfiles PetFinder preparados;
- probabilidades históricas precalculadas para perfiles de demostración;
- explicaciones locales preparadas para Protectora;
- embeddings DINOv2 de PetFinder;
- prototipos visuales derivados de Tsinghua Dogs;
- recursos generados para el análisis textual y las nubes de palabras.

Esta separación permite mantener el entrenamiento y la evaluación científica fuera del flujo de navegación de la aplicación.


## 4. Flujo funcional de los dos perfiles

### 4.1. Recorrido Adoptante

El recorrido del adoptante permite:

- definir preferencias sobre características del perro;
- consultar candidatos compatibles dentro del catálogo histórico preparado;
- acceder a fichas individuales;
- marcar favoritos;
- utilizar una imagen de referencia para recuperar perros visualmente similares;
- explorar resultados visuales sin interpretar la similitud como una probabilidad de raza.

Para la búsqueda fotográfica, la arquitectura prevista es:

```text
Navegador
   -> API server-side de Next.js
   -> servicio Python DINOv2
   -> embedding de 384 dimensiones
   -> similitud frente a embeddings PetFinder preparados
   -> candidatos ordenados
   -> respuesta al navegador
```

### 4.2. Recorrido Protectora

La Vista Protectora incorpora:

- acceso autenticado mediante una cuenta de demostración;
- perfiles históricos preparados;
- probabilidad de adopción lenta derivada del análisis complementario de PetFinder;
- explicaciones locales previamente calculadas;
- indicadores de completitud;
- resultados del análisis textual;
- visualizaciones orientadas a interpretar factores asociados a una adopción más lenta o más sencilla.

Las credenciales y el secreto de sesión no se almacenan en el repositorio. Se suministran como variables de entorno del servicio de Azure.


## 5. Despliegue y validación

El despliegue del frontend se automatiza mediante el workflow:

`/.github/workflows/main_invisibledogs-predict.yml`

Cuando se actualiza la rama `main`, GitHub Actions ejecuta de forma secuencial:

1. checkout del repositorio;
2. instalación de Node.js;
3. `npm ci`;
4. lint;
5. pruebas automatizadas;
6. build de Next.js;
7. preparación del paquete de Azure;
8. autenticación en Azure mediante identidad federada;
9. despliegue en Azure App Service.

El workflow trabaja específicamente sobre `frontend/`, de forma que los notebooks, modelos de entrenamiento, datasets y documentación científica pueden mantenerse en el mismo repositorio sin formar parte del paquete web desplegado.

La versión actual del frontend ha sido validada en Azure mediante comprobaciones de las rutas principales, autenticación de Protectora y carga de los recursos de Text Mining.


## 6. Estado actual del runtime y limitaciones

Es importante distinguir entre **artefactos científicos disponibles** y **componentes actualmente ejecutados en tiempo real por la web**.

### Operativo en el frontend desplegado

- navegación de Adoptante y Protectora;
- autenticación de demostración de Protectora;
- consulta de perfiles históricos preparados;
- filtros y ranking sobre artefactos precalculados;
- visualizaciones de Text Mining;
- uso de explicaciones y probabilidades históricas preparadas.

### Implementado en el repositorio, pero pendiente de despliegue independiente

- servicio Python `services/dinov2-inference/` para generar embeddings DINOv2 de fotografías nuevas.

### Pendiente de integración en el runtime web

- inferencia en tiempo real del **modelo principal de Austin Animal Center** sobre nuevos registros;
- generación dinámica de explicaciones asociadas a nuevas predicciones;
- integración del modelo complementario de PetFinder como servicio de inferencia cuando se requiera una predicción nueva.

Por tanto, el MVP desplegado ya demuestra la arquitectura, los recorridos y la reutilización de artefactos validados, pero la integración completa de los modelos predictivos para nuevas entradas constituye la siguiente fase técnica del proyecto.


## 7. Trazabilidad con el resto del TFM

La relación entre los principales componentes científicos y el MVP es la siguiente:

| Componente | Procedencia | Uso en el MVP |
|---|---|---|
| Modelo principal de larga estancia | Austin Animal Center | Núcleo predictivo del TFM; integración runtime pendiente |
| Modelo complementario de adopción lenta | PetFinder | Probabilidades y análisis complementarios; integración dinámica pendiente |
| Interpretabilidad | Modelización + SHAP | Explicaciones preparadas para perfiles de demostración |
| Regresión de duración de estancia | Análisis complementario | Resultado científico, no utilizado como score combinado |
| DINOv2 PetFinder | Análisis visual | Recuperación por similitud visual |
| Tsinghua Dogs | Validación externa visual | Referencia externa y prototipos visuales |
| Text Mining PetFinder | Análisis complementario | Nubes de palabras y términos diferenciales |

Esta separación evita mezclar objetivos predictivos distintos y mantiene la trazabilidad entre los notebooks experimentales y los componentes incorporados al producto demostrador.
